# Solving the 1D Transient Heat Equation with Physics-Informed Neural Networks (PINNs)

This notebook solves the 1D heat equation using physics-informed neural networks (PINNs) with the following setup:

- **Domain**: $\Omega = [0, 1] \text{ and } t = [0, 1]$
- **Boundary Conditions**: $u(x = 0, t) = u(x = 1, t) = 0$
- **Initial Conditions**: $u(x, t = 0) = \sin(\pi x)$
- **Analytical Solution**: $u_{\text{analytical}} = e^{-\alpha \pi^2 t} \sin(\pi x)$

1. Import necessary libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

2. Define parameters

In [ ]:
# Hyperparameters
layers = [2, 50, 50, 50, 1]  # 2 input (x, t), 3 hidden layers with 50 neurons, 1 output (u)
alpha = 0.1  # Thermal diffusivity
learning_rate = 0.001
epochs = 1000
beta1 = 1.0  # Scaling factor for initial condition loss
beta2 = 1.0  # Scaling factor for boundary condition loss

# Define the range of x and specific timesteps to plot
x_test = torch.linspace(0, 1, 100).reshape(-1, 1)
timesteps = [0, 25, 50, 75, 99]  # Different timesteps to plot

# Generate collocation points
x_collocation = torch.linspace(0, 1, 10).reshape(-1, 1)
# For a skewed distribution, you can use a non-linear spacing like this:
# s = torch.linspace(0, 1, 10)
# x_skew = 0.5 * (1 - torch.cos(torch.pi * s)).reshape(-1, 1)
t_collocation = torch.linspace(0, 1, 10).reshape(-1, 1)
x_collocation, t_collocation = torch.meshgrid(x_collocation.squeeze(), t_collocation.squeeze(), indexing='ij')
x_collocation = x_collocation.reshape(-1, 1)
t_collocation = t_collocation.reshape(-1, 1)

# Ensure requires_grad is True for collocation points
x_collocation.requires_grad = True
t_collocation.requires_grad = True

3. Define analytical solution and plot

In [ ]:
def analytical_solution(x, t, alpha):
    return torch.exp(-alpha * torch.pi**2 * t) * torch.sin(torch.pi * x)

# Plot Analytical solution
plt.figure(figsize=(10, 4))
for n in timesteps:
    t_test = torch.full((100, 1), n / 99)  # Create a tensor with the same time value
    u_exact = analytical_solution(x_test, t_test, alpha).detach().numpy()
    plt.plot(x_test.numpy(), u_exact, label=f'Analytical t={n / 99:.2f}')

# Plot collocation points
plt.scatter(x_collocation.detach().numpy(), np.zeros_like(x_collocation.detach().numpy()), color='red', s=10, label='Collocation Points')

plt.xlabel('x')
plt.ylabel('u(x, t)')
plt.title('1D Heat Equation Solution (Analytical)')
plt.legend()
plt.show()

3. Define Neural Network

In [ ]:
class HeatPINN(nn.Module):
    def __init__(self, layers):
        super(HeatPINN, self).__init__()
        self.layers = nn.ModuleList()

        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i + 1]))

    def forward(self, x, t):
        X = torch.cat([x, t], dim=1)
        for i in range(len(self.layers) - 1):
            X = torch.tanh(self.layers[i](X))
        X = self.layers[-1](X)
        return X
    
# Initialize model and optimizer
model = HeatPINN(layers)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

4. Define the PDE loss function

In [ ]:
def pde_loss(model, x, t, alpha):
    x.requires_grad = True
    t.requires_grad = True

    u = model(x, t)
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]

    # Residual of the PDE
    f = u_t - alpha * u_xx
    return torch.mean(f ** 2)

5. Define initial and boundary conditions

In [ ]:
def initial_condition(x):
    return torch.sin(torch.pi * x)

def boundary_condition(x, t):
    return torch.zeros_like(t)

6. Solve the PDE using PINNs

In [ ]:
losses = []
# Training loop
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    # Compute loss for initial condition with only 10 points
    x_init = torch.linspace(0, 1, 10).reshape(-1, 1)
    t_init = torch.zeros_like(x_init)
    u_init = initial_condition(x_init)
    u_pred = model(x_init, t_init)
    loss_init = torch.mean((u_pred - u_init) ** 2)

    # Compute loss for boundary conditions with only 2 points
    x_bc = torch.tensor([[0.0], [1.0]])
    t_bc = torch.tensor([[0.0], [1.0]])
    u_bc = boundary_condition(x_bc, t_bc)
    u_pred_bc = model(x_bc, t_bc)
    loss_bc = torch.mean((u_pred_bc - u_bc) ** 2)

    # Compute PDE loss
    loss_pde = pde_loss(model, x_collocation, t_collocation, alpha)

    # Total loss with scaling factors
    loss = beta1 * loss_init + beta2 * loss_bc + loss_pde
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    # Output loss and plot every 100 epochs
    if epoch % 100 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

        # Evaluate the model and plot the solution for different timesteps
        model.eval()

        # Define the range of x and specific timesteps to plot
        x_test = torch.linspace(0, 1, 100).reshape(-1, 1)
        timesteps = [0, 25, 50, 75, 99]  # Different timesteps to plot

        plt.figure(figsize=(10, 4))
        for n in timesteps:
            t_test = torch.full((100, 1), n / 99)  # Create a tensor with the same time value
            u_pred = model(x_test, t_test).detach().numpy()
            plt.plot(x_test.numpy(), u_pred, label=f'PINN t={n / 99:.2f}')

        # Plot collocation points
        plt.scatter(x_collocation.detach().numpy(), np.zeros_like(x_collocation.detach().numpy()), color='red', s=10, label='Collocation Points')

        plt.xlabel('x')
        plt.ylabel('u(x, t)')
        plt.title(f'1D Heat Equation Solution with PINNs (Epoch {epoch})')
        plt.legend()
        plt.show()

In [ ]:
# Updated training loop: log components, clip gradients, add scheduler
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # lower LR
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=50)

losses = []
losses_init = []
losses_bc = []
losses_pde = []

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    # Initial condition loss
    x_init = torch.linspace(0, 1, 10).reshape(-1, 1)
    t_init = torch.zeros_like(x_init)
    u_init = initial_condition(x_init)
    u_pred_init = model(x_init, t_init)
    loss_init = torch.mean((u_pred_init - u_init) ** 2)

    # Boundary condition loss
    x_bc = torch.tensor([[0.0], [1.0]])
    t_bc = torch.linspace(0, 1, 2).reshape(-1,1)  # better: times across [0,1]
    u_bc = boundary_condition(x_bc, t_bc)
    u_pred_bc = model(x_bc, t_bc)
    loss_bc = torch.mean((u_pred_bc - u_bc) ** 2)

    # PDE loss
    loss_pde = pde_loss(model, x_collocation, t_collocation, alpha)

    # Total loss with scaling factors
    loss = beta1 * loss_init + beta2 * loss_bc + loss_pde
    loss.backward()

    # Gradient clipping to avoid large gradient steps
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    # record components
    losses.append(loss.item())
    losses_init.append(loss_init.item())
    losses_bc.append(loss_bc.item())
    losses_pde.append(loss_pde.item())

    # scheduler step on total loss (or on validation)
    scheduler.step(loss.item())

    # Output loss and plot every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss={loss.item():.3e}, init={loss_init.item():.3e}, bc={loss_bc.item():.3e}, pde={loss_pde.item():.3e}")

        # Evaluate the model and plot the solution for different timesteps
        model.eval()

        # Define the range of x and specific timesteps to plot
        x_test = torch.linspace(0, 1, 100).reshape(-1, 1)
        timesteps = [0, 25, 50, 75, 99]  # Different timesteps to plot

        plt.figure(figsize=(10, 4))
        for n in timesteps:
            t_test = torch.full((100, 1), n / 99)  # Create a tensor with the same time value
            u_pred = model(x_test, t_test).detach().numpy()
            plt.plot(x_test.numpy(), u_pred, label=f'PINN t={n / 99:.2f}')

        # Plot collocation points
        plt.scatter(x_collocation.detach().numpy(), np.zeros_like(x_collocation.detach().numpy()), color='red', s=10, label='Collocation Points')

        plt.xlabel('x')
        plt.ylabel('u(x, t)')
        plt.title(f'1D Heat Equation Solution with PINNs (Epoch {epoch})')
        plt.legend()
        plt.show()

7. Calculate error

In [ ]:
# Plot the absolute error
plt.figure(figsize=(10, 4))
for n in timesteps:
    t_test = torch.full((100, 1), n / 99)
    u_pred = model(x_test, t_test).detach().numpy()
    u_exact = analytical_solution(x_test, t_test, alpha).detach().numpy()
    error = np.abs(u_pred - u_exact)

    plt.plot(x_test.numpy(), error, label=f'Error at t={n / 99:.2f}')

# Plot collocation points
plt.scatter(x_collocation.detach().numpy(), np.zeros_like(x_collocation.detach().numpy()), color='red', s=10, label='Collocation Points')

plt.xlabel('x')
plt.ylabel('Absolute Error')
plt.title('Absolute Error between PINN and Analytical Solution')
plt.legend()
plt.show()

# Plot training loss history
plt.figure(figsize=(10, 4))
plt.plot(range(len(losses)), losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss History')
plt.yscale('log')  # Use log scale for better visualization
plt.grid(True)
plt.show()